# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [1]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    # Control de errores: si el solapamiento es mayor o igual que el tamaño del bloque,
    # el bucle se volvería infinito. Lo evitamos con esta validación matemática.
    if overlap >= chunk_size:
        raise ValueError("El 'overlap' debe ser estrictamente menor que el 'chunk_size'.")

    chunks = []
    start = 0
    text_len = len(text)

    # Recorremos el texto mientras el índice de inicio no haya llegado al final
    while start < text_len:
        # El final del trozo actual es el inicio más el tamaño del bloque solicitado
        end = start + chunk_size

        # Extraemos el fragmento de texto utilizando slicing
        chunk = text[start:end]
        chunks.append(chunk)

        # Calculamos el siguiente punto de inicio aplicando el salto con solapamiento
        # Salto real = chunk_size - overlap
        start += (chunk_size - overlap)

        # Condición de salida segura: si el siguiente bloque empieza más allá del texto,
        # o si el bloque actual ya llegó al final exacto del texto, rompemos el bucle.
        if end >= text_len:
            break

    return chunks


# =====================================================================
# COMPROBACIÓN PRÁCTICA CON EL TEXTO DE JUGUETE
# =====================================================================

# Creamos un texto dummy simulado (500 repeticiones de la palabra 'word ')
# Longitud total: 500 * 5 = 2500 caracteres
long_text = "word " * 500

# Ejecutamos nuestra función con los parámetros por defecto
resultado_chunks = chunk_text(long_text, chunk_size=200, overlap=40)

print("--- VERIFICACIÓN DEL CHUNKER ---")
print(f"Longitud total del texto original: {len(long_text)} caracteres.")
print(f"Número total de chunks generados:  {len(resultado_chunks)}")

print("\n--- INSPECCIÓN DE LOS PRIMEROS CHUNKS ---")
print(f"Chunk 0 (Longitud {len(resultado_chunks[0])}): '{resultado_chunks[0][:60]}...'")
print(f"Chunk 1 (Longitud {len(resultado_chunks[1])}): '{resultado_chunks[1][:60]}...'")

# Validamos que haya solapamiento real: el final del chunk 0 debe coincidir con el inicio del chunk 1
final_chunk_0 = resultado_chunks[0][-40:]
inicio_chunk_1 = resultado_chunks[1][:40]
print(f"\n¿El solapamiento es idéntico entre bloques contiguos?: {final_chunk_0 == inicio_chunk_1}")


--- VERIFICACIÓN DEL CHUNKER ---
Longitud total del texto original: 2500 caracteres.
Número total de chunks generados:  16

--- INSPECCIÓN DE LOS PRIMEROS CHUNKS ---
Chunk 0 (Longitud 200): 'word word word word word word word word word word word word ...'
Chunk 1 (Longitud 200): 'word word word word word word word word word word word word ...'

¿El solapamiento es idéntico entre bloques contiguos?: True


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [3]:
!pip install faiss-cpu
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Reutilizamos la función de chunking del ejercicio anterior para crear datos reales de prueba
def chunk_text(text: str, chunk_size: int = 150, overlap: int = 20) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
        if end >= len(text): break
    return chunks

# 1. Definimos un documento temático largo para segmentar
documento_rag = (
    "The Pyramids of Giza were built during the Old Kingdom of Egypt. "
    "Pharaoh Khufu commissioned the Great Pyramid around 2560 BC as a monumental tomb. "
    "Deep inside the structure lies the King's Chamber, holding a massive granite sarcophagus. "
    "In modern times, archaeologists use advanced cosmic-ray muon radiography to scan for hidden voids. "
    "NASA's Artemis program aims to land the next humans on the Moon using the SLS rocket and Orion spacecraft. "
    "The lunar south pole is highly valuable due to the presence of water ice in permanently shadowed craters. "
    "SpaceX is developing Starship as a fully reusable transportation system designed for Mars missions."
)

# Segmentamos el texto en trozos pequeños (chunks)
chunks_texto = chunk_text(documento_rag, chunk_size=120, overlap=15)

# 2. Cargar el modelo y generar los embeddings para cada fragmento
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_chunks = model.encode(chunks_texto)

# Conversión crítica para FAISS: exige matrices de tipo float32 en NumPy
vectors_np = np.array(embeddings_chunks).astype("float32")

# 3. Construir el índice vectorial de FAISS (Usando Distancia L2)
dimension = vectors_np.shape[1]  # En all-MiniLM-L6-v2 es de 384 dimensiones
index = faiss.IndexFlatL2(dimension)

# Añadimos los vectores al índice
index.add(vectors_np)

# 4. Definir una consulta (Query) y transformarla en embedding
query = "Tell me about space exploration and going to the Moon or Mars."
query_embedding = model.encode([query])
query_vector_np = np.array(query_embedding).astype("float32")

# 5. Realizar la búsqueda en el índice para recuperar el Top-3 de fragmentos más cercanos
k = 3  # Número de resultados a recuperar
distancias, indices = index.search(query_vector_np, k)

# 6. Mostrar los resultados de la recuperación
print("--- MOTOR DE BÚSQUEDA SEMÁNTICA CON FAISS ---")
print(f"Query del usuario: '{query}'\n")
print(f"Total de chunks indexados: {index.ntotal}")
print("----------------------------------------------------------------")
print("TOP-3 RECONSTRUIDO POR SIMILITUD VECTORIAL:\n")

for i, (idx, dist) in enumerate(zip(indices[0], distancias[0])):
    # Recuperamos el texto original mapeándolo con el índice devuelto por FAISS
    texto_recuperado = chunks_texto[idx]
    print(f"Resultado #{i+1} [Índice Chunk: {idx}] [Distancia L2: {dist:.4f}]:")
    print(f" > '{texto_recuperado.strip()}'\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- MOTOR DE BÚSQUEDA SEMÁNTICA CON FAISS ---
Query del usuario: 'Tell me about space exploration and going to the Moon or Mars.'

Total de chunks indexados: 7
----------------------------------------------------------------
TOP-3 RECONSTRUIDO POR SIMILITUD VECTORIAL:

Resultado #1 [Índice Chunk: 6] [Distancia L2: 0.8540]:
 > 'for Mars missions.'

Resultado #2 [Índice Chunk: 3] [Distancia L2: 1.0833]:
 > 'an for hidden voids. NASA's Artemis program aims to land the next humans on the Moon using the SLS rocket and Orion spac'

Resultado #3 [Índice Chunk: 5] [Distancia L2: 1.2494]:
 > 'ently shadowed craters. SpaceX is developing Starship as a fully reusable transportation system designed for Mars missio'



## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Definir la consulta del usuario (Query)
query = "What is the primary function of the mitochondria in a cell?"

# 2. Definir los tres documentos sintéticos (1 relevante y 2 que actúan como ruido)
doc_revalent = "The mitochondria acts as the powerhouse of the cell, generating adenosine triphosphate (ATP) through cellular respiration."
doc_noise_1  = "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France."
doc_noise_2  = "Stock markets experienced high volatility today as investors analyzed the latest inflation data released by the central bank."

documentos = [doc_revalent, doc_noise_1, doc_noise_2]
nombres_docs = ["Documento Relevante (Biología)", "Ruido 1 (Geografía/Historia)", "Ruido 2 (Economía)"]

# 3. Inicializar el modelo y generar los embeddings vectoriales
model = SentenceTransformer("all-MiniLM-L6-v2")

query_embedding = model.encode(query)
docs_embeddings = model.encode(documentos)

# 4. Implementar la función de Similitud Coseno pura con NumPy
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 5. Calcular los scores de similitud cruzando la query con cada documento
resultados_ranking = []
for i in range(len(documentos)):
    score = cosine_similarity(query_embedding, docs_embeddings[i])
    resultados_ranking.append({
        "nombre": nombres_docs[i],
        "texto": documentos[i],
        "score": score
    })

# 6. Ordenar el ranking de mayor a menor score de similitud
# Explicación: sorted ordena de menor a mayor por defecto, usando reverse=True conseguimos el Top-1 arriba
ranking_ordenado = sorted(resultados_ranking, key=lambda x: x["score"], reverse=True)

# 7. Mostrar los resultados por pantalla
print("--- EVALUACIÓN MANUAL DE RANKING RAG ---")
print(f"Query: '{query}'\n")

print("RANKING FINAL GENERADO POR EL MODELO:")
print("----------------------------------------------------------------")
for posicion, res in enumerate(ranking_ordenado, start=1):
    print(f"Puesto #{posicion} | {res['nombre']} | Similitud Coseno: {res['score']:.4f}")
    print(f" > Content: '{res['texto']}'\n")

# 8. Verificación automatizada por código (Assert)
# Comprobamos que el primer elemento de la lista ordenada sea efectivamente nuestro documento biológico relevante
assert ranking_ordenado[0]["nombre"] == "Documento Relevante (Biología)", "¡Error! El sistema no priorizó el documento correcto."
print("¡VERIFICACIÓN EXITOSA! El documento relevante ha quedado en el puesto #1 del ranking con el score más alto.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- EVALUACIÓN MANUAL DE RANKING RAG ---
Query: 'What is the primary function of the mitochondria in a cell?'

RANKING FINAL GENERADO POR EL MODELO:
----------------------------------------------------------------
Puesto #1 | Documento Relevante (Biología) | Similitud Coseno: 0.6439
 > Content: 'The mitochondria acts as the powerhouse of the cell, generating adenosine triphosphate (ATP) through cellular respiration.'

Puesto #2 | Ruido 1 (Geografía/Historia) | Similitud Coseno: 0.1899
 > Content: 'The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France.'

Puesto #3 | Ruido 2 (Economía) | Similitud Coseno: -0.0106
 > Content: 'Stock markets experienced high volatility today as investors analyzed the latest inflation data released by the central bank.'

¡VERIFICACIÓN EXITOSA! El documento relevante ha quedado en el puesto #1 del ranking con el score más alto.
